In [1]:
!pip install openai-whisper faster-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 12.0 MB/s eta 0:00:00a 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 59.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 52.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 89.5 MB/s eta 0:00:00:00:0100:01
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=bcb603da06ce96760a2d656c2607c470e55e7fbe11f4886c00c17c0ac895d3ed
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [ ]:
import os
import json
import requests
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
import torch

CACHE_DIR = "./cache"
AUDIO_DIR = os.path.join(CACHE_DIR, "audio")
TRANS_DIR = os.path.join(CACHE_DIR, "transcripts")

os.makedirs(AUDIO_DIR, exist_ok=True)
os.makedirs(TRANS_DIR, exist_ok=True)

CSV_PATH = "/kaggle/input/datasets/mitishraina/data-joshtalks/data.csv"

df = pd.read_csv(CSV_PATH)

OLD_PREFIX = "https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi"
NEW_PREFIX = "https://storage.googleapis.com/upload_goai"

def fix_url(url):
    if pd.isna(url):
        return None

    if "upload_goai" in url:
        return url

    if "joshtalks-data-collection" in url:
        return url.replace(OLD_PREFIX, NEW_PREFIX)

    if url.startswith("joshtalks-data-collection"):
        return NEW_PREFIX + "/" + url.split("/", 1)[1]

    return url

df["rec_url_fixed"] = df["rec_url_gcp"].apply(fix_url)
df["transcription_url_fixed"] = df["transcription_url_gcp"].apply(fix_url)

def download_file(url, save_path):
    if not url:
        return None

    if os.path.exists(save_path):
        return save_path

    try:
        r = requests.get(url, timeout=10)
        if r.status_code != 200:
            return None

        if "audio" not in r.headers.get("Content-Type", ""):
            return None

        with open(save_path, "wb") as f:
            f.write(r.content)

        return save_path

    except:
        return None


def get_audio_path(row):
    path = os.path.join(AUDIO_DIR, f"{row.recording_id}.wav")
    return download_file(row.rec_url_fixed, path)


def get_transcript(row):
    path = os.path.join(TRANS_DIR, f"{row.recording_id}.json")

    if not os.path.exists(path):
        try:
            r = requests.get(row.transcription_url_fixed, timeout=10)
            if r.status_code != 200:
                return ""

            data = r.json()
            with open(path, "w") as f:
                json.dump(data, f)

        except:
            return ""

    try:
        data = json.load(open(path))
        return " ".join([seg["text"] for seg in data])
    except:
        return ""


def is_valid_audio(path):
    return path and os.path.exists(path) and os.path.getsize(path) > 10000


def process_row(row):
    return get_audio_path(row), get_transcript(row)

results = []
with ThreadPoolExecutor(max_workers=16) as ex:
    futures = [ex.submit(process_row, row) for _, row in df.iterrows()]
    for f in tqdm(futures):
        results.append(f.result())

df["audio_path"] = [r[0] for r in results]
df["reference"] = [r[1] for r in results]

df = df[df["audio_path"].apply(is_valid_audio)].reset_index(drop=True)



print("Detecting GPUs...")
num_gpus = torch.cuda.device_count()
print(f"GPUs available: {num_gpus}")

if num_gpus == 0:
    raise RuntimeError("No GPU found. Make sure Kaggle accelerator is set to GPU.")

def transcribe_on_gpu(gpu_id, paths):
    from faster_whisper import WhisperModel

    model = WhisperModel(
        "small",
        device="cuda",
        device_index=gpu_id,
        compute_type="float16",
        num_workers=2  
    )

    outputs = []

    for path in tqdm(paths, desc=f"GPU {gpu_id}"):
        try:
            segments, _ = model.transcribe(
                path,
                language="hi",
                beam_size=1,
                vad_filter=True
            )
            text = " ".join([s.text for s in segments])
            outputs.append(text)
        except Exception:
            outputs.append("")

    return outputs


def split_work(paths, n):
    return [paths[i::n] for i in range(n)]

paths = df["audio_path"].tolist()
chunks = split_work(paths, num_gpus)

with ProcessPoolExecutor(max_workers=num_gpus) as ex:
    futures = [
        ex.submit(transcribe_on_gpu, i, chunk)
        for i, chunk in enumerate(chunks)
    ]

results = []
for f in futures:
    results.extend(f.result())

df["asr_raw"] = results

num_map = {
    "शून्य":0, "एक":1, "दो":2, "तीन":3, "चार":4, "पांच":5,
    "छह":6, "सात":7, "आठ":8, "नौ":9, "दस":10, "बीस":20,
    "तीस":30, "चालीस":40, "पचास":50, "साठ":60,
    "सत्तर":70, "अस्सी":80, "नब्बे":90, "सौ":100, "हज़ार":1000
}

def parse_number_phrase(tokens):
    total, current = 0, 0
    for t in tokens:
        val = num_map.get(t)
        if val:
            if val in [100, 1000]:
                current *= val
            else:
                current += val
    return total + current


def normalize_numbers(text):
    tokens = text.split()
    result, buffer = [], []

    for t in tokens:
        if t in num_map:
            buffer.append(t)
        else:
            if buffer:
                result.append(str(parse_number_phrase(buffer)))
                buffer = []
            result.append(t)

    if buffer:
        result.append(str(parse_number_phrase(buffer)))

    return " ".join(result)

df["num_norm"] = [normalize_numbers(x) for x in df["asr_raw"]]

english_loanwords = {
    "इंटरव्यू", "जॉब", "प्रॉब्लम", "कंप्यूटर", "मोबाइल",
    "नेटवर्क", "सिस्टम", "फाइल", "डेटा", "कोड"
}

def detect_english(text):
    return " ".join(
        f"[EN]{t}[/EN]" if t in english_loanwords else t
        for t in text.split()
    )

df["tagged"] = [detect_english(x) for x in df["num_norm"]]

df.to_csv("q2_output.csv", index=False)

print("Saved to q2_output.csv")

100%|██████████| 104/104 [00:00<00:00, 2847.38it/s]

Detecting GPUs...
GPUs available: 2



GPU 0: 100%|██████████| 52/52 [1:04:44<00:00, 74.70s/it] 


Saved to q2_output.csv


In [ ]:
import shutil

shutil.make_archive("joshtalks_cache", "zip", "/kaggle/working/cache")

print("Zip file created: joshtalks_cache.zip")

Zip file created: joshtalks_cache.zip
